In [0]:
import pandas as pd
import hashlib

In [0]:


CHUNK_WORD_LIMIT = 300
CHUNK_SIZE = 250
CHUNK_OVERLAP = 40

def hash_id(raw_id: str) -> str:
    return hashlib.sha256(str(raw_id).encode()).hexdigest()[:16]

def chunk_text(text: str):
    words = text.split()
    if len(words) <= CHUNK_WORD_LIMIT:
        return [text]
    chunks = []
    start = 0
    while start < len(words):
        end = start + CHUNK_SIZE
        chunks.append(" ".join(words[start:end]))
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return chunks

def run(silver: pd.DataFrame) -> pd.DataFrame:
    df = silver[silver["reviewText_clean"].str.split().str.len() >= 8].copy()  # drop near-empty (<8 words), low signal
    df["reviewer_hash"] = df["reviewerID"].apply(hash_id)

    rows = []
    for _, r in df.iterrows():
        chunks = chunk_text(r["reviewText_clean"])
        for i, chunk in enumerate(chunks):
            doc_text = (
                f"Product: {r['asin']}. Rating: {r['overall']}/5. "
                f"Summary: {r['summary_clean']}. Review: {chunk}"
            )
            rows.append({
                "doc_id": f"{r['reviewer_hash']}_{r['asin']}_{i}",
                "reviewer_hash": r["reviewer_hash"],
                "asin": r["asin"],
                "overall": r["overall"],
                "chunk_index": i,
                "n_chunks": len(chunks),
                "document_text": doc_text,
            })

    rag_docs = pd.DataFrame(rows)
    rag_docs.to_pickle("rag_documents.pkl")
    n_split_reviews = (rag_docs.drop_duplicates(subset=["reviewer_hash","asin"])["n_chunks"] > 1).sum()
    print(f"[rag_prep] {len(df)} reviews -> {len(rag_docs)} chunks "
          f"({n_split_reviews} reviews were split into multiple chunks, "
          f"{n_split_reviews/len(df)*100:.1f}%)")
    return rag_docs

if __name__ == "__main__":
    silver = pd.read_csv("/Workspace/Users/vertubouka@gmail.com/github-databricks/Generative AI project/DbAcademy/silver_table.csv")
    silver["reviewText_clean"] = silver["reviewText"].fillna("").str.strip().str.lower()
    silver["summary_clean"] = silver["summary"].fillna("").str.strip().str.lower()
    run(silver)